# Timestream Lambda Function Sample Application

This notebook demonstrates generating data, according to a schema defined by the user; deploying an AWS Lambda function to process it; and visualizing the data using Grafana.

## Imports

Imports necessary for all steps.

In [1]:
import boto3
from datetime import datetime, timedelta, timezone
import json
import math
import sys

sys.path.append('../src/')
from lambda_sample import data_generator, lambda_helper, grafana_helper

# AWS Settings

These constants and variables are used throughout this notebook to configure writes to Timestream for LiveAnalytics and the generation of Amazon Managed Grafana dashboards.

Adjust these settings as needed.

In [3]:
DATABASE_NAME = "sample_app_database"
TABLE_NAME = "sample_app_table"

# To be used later, by the Lambda function, to create the Timestream for LiveAnalytics table.
# If you created your table manually, update with the actual values you configured for your table.
# Default values when creating a new table in the AWS console.
MEM_STORE_RETENTION_PERIOD_IN_HOURS = 12
MAG_STORE_RETENTION_PERIOD_IN_DAYS = 3653 # 10 years

# The number of records to ingest to Timestream for LiveAnalytics at a time.
# Timestream for LiveAnalytics accepts a maximum of 100 records at a time.
BATCH_SIZE = 100
# BATCH_SIZE = 1

# The precision of the timestamp for each generated record. Valid options are "MILLISECONDS", "SECONDS", and "MICROSECONDS".
# This will also be included as a query parameter in the request sent to the Lambda function.
PRECISION = "MICROSECONDS"

session = boto3.session.Session()

## Step 1: Generate Data

### Generate Data

The data generator classes use the `generate` function to generate data. The arguments to `generate` are as follows:
- `start_date`: The start date to use when generating records. This cannot be older in hours than the memory retention period in hours value for the table.
- `end_date`: The end date to use when generating records. The maximum end date Timestream for LiveAnalytics allows is 15 minutes in the future.
- `reporting_frequency`: The frequency that records are generated by all entities, for example, every 2 seconds, every 5 hours, etc.
- `num_entities` The number of entities that will report for each timestamp, for example, the number of servers or number of stocks.
- `precision`: The precision to use for record timestamps. Valid options are `"MILLISECONDS"`, `"SECONDS"`, and `"MICROSECONDS"`.
- `generate_unique_options_fallback`: Whether to generate random strings for dimension values after all values in a dimension template's "unique_options" array have been used.

The following data generators are available:
- `DevOpsDataGenerator`: Generates generic DevOps time series data for servers.
- `IoTDateGenerator`: Generates generic IoT time series data for devices.
- `StockMarketGenerator`: Generates time series data simulating stock market prices.
- `WeatherDataGenerator`: Generates time series data simulating weather reporting for different US cities.
- `GamingDataGenerator`: Generates time series data simulating player activity in a competitive online video game.
- `AirQualityDataGenerator`: Generates time series data simulating air quality in different cities around the world.
- `PatientDataGenerator`: Generates time series data simulating the status of healthcare patients.
- `EnergyDataGenerator`: Generates time series data simulating building energy usage.
- `FlightDataGenerator`: Generates time series data simulating different airline flights and the status of in-flight planes.
- `ExchangeRateDataGenerator`: Generates time series data simulating the fluctuating exchange rates of different currency pairs.
- `CustomDataGenerator`: Allows users to define their own `measure_templates` and `dimension_templates` to generate data of their choosing.

In [ ]:
# All timestamps default to UTC
end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(hours=2)
reporting_frequency = timedelta(minutes=1)
num_entities = 10

if end_date > datetime.now(timezone.utc) + timedelta(minutes=15):
    raise Exception("The end date for data generation cannot be more than 15 minutes in the future")
if start_date < datetime.now(timezone.utc) - timedelta(hours=MEM_STORE_RETENTION_PERIOD_IN_HOURS):
    raise Exception(f"The start date for data generation cannot be more than {MEM_STORE_RETENTION_PERIOD_IN_HOURS} hours in the past")
if start_date >= end_date:
    raise Exception("The start date and end date for data generation are the same")
if (end_date - start_date) < reporting_frequency:
    raise Exception("The reporting frequency is too small for the data generation time range")

# By default, generate DevOps data, which simulates reporting from servers
# Define data_generator to help generate Grafana dashboard later
data_generator = data_generator.DevOpsDataGenerator()
sample_data = data_generator.generate(start_date, end_date, reporting_frequency, num_entities, precision=PRECISION)

# Custom data
#measure_templates = [
#    {
#        "name": "exchange_rate",
#        "type": "DOUBLE",
#        "max_variation": 1.0,
#        "max": 90.0,
#        "min": 0.62
#    }
#]

#dimension_templates = [
#    {
#        "name": "currency_pair",
#        "value_length": 4,
#        "unique_options": ["USD/EUR", "USD/CAD", "USD/GPP", "USD/CNY", "GBP/CAD", "GBP/JPY", "GBP/INR", "CHF/INR", "XAU/CNY", "UYU/CAD"]
#    }
#]

#data_generator = CustomDataGenerator(measure_templates=measure_templates, dimension_templates=dimension_templates)
#sample_data = data_generator.generate(start_date, end_date, reporting_frequency, num_entities, precision=PRECISION)

# Print generated data
print(json.dumps(sample_data, indent=2))

## Step 2: Calculate Cost Metrics

The following cell provides metrics that can be input into the [AWS pricing calculator](https://calculator.aws/#/) to give an estimate of costs for ingesting data to Timestream for LiveAnalytics.

The metrics are:

- Memory store writes.
    - This is calculated by determining the number of records that would be ingested within the `MEM_STORE_RETENTION_PERIOD_IN_HOURS` time frame.

Magnetic store writes are not calculated since Timestream for LiveAnalytics does not allow ingesting records with timestamps outside of the `MEM_STORE_RETENTION_PERIOD_IN_HOURS` time frame. In order for records to be stored in magnetic storage, they need to first be stored in memory then be moved to magnetic storage once enough time has passed.

These cost metrics may not be accurate, as there may be a delay between generating the data and ingesting it, causing some amounts of records to be put into magnetic store or rejected due to being too old.

In [ ]:
# The AWS pricing calculator only allows per second, per minute, per hour, per day, and per month.

num_records = len(sample_data)
time_diff = end_date - start_date

if time_diff <= timedelta(seconds=1):
    unit = "second"
    scaled_count = num_records
elif time_diff <= timedelta(minutes=1):
    unit = "minute"
    scaled_count = num_records
elif time_diff <= timedelta(hours=1):
    unit = "hour"
    scaled_count = num_records
elif time_diff <= timedelta(days=1):
    unit = "day"
    scaled_count = num_records
# 30 days in a month is standard for billing
elif time_diff <= timedelta(days=30):
    unit = "month"
    scaled_count = num_records
else:
    unit = "month"
    print(time_diff.days)
    # Round up, since the AWS pricing calculator does not accept decimal numbers
    scaled_count = math.ceil(num_records / (time_diff.days / 30))

print(f"Total records: {num_records}")
print(f"Memory store writes: {scaled_count} per {unit}")

## Step 3: Deploy AWS Lambda Function

The following code will construct and deploy a Lambda function that ingests data to Timestream.

### Generate and Deploy Lambda Function

In [ ]:
lambda_url = lambda_helper.create_lambda(session, lambda_name="TimestreamSampleLambda",
                                         database_name=DATABASE_NAME, table_name=TABLE_NAME,
                                         role_name="TimestreamSampleRole")

## Step 4: Send Data to the Lambda Function

The following code will send the generated sample data to the Lambda function's URL with SigV4 authenticated requests, ensuring requests do not exceed Lambda's limit of 6 MB.

### Send Generated Data

In [ ]:
lambda_helper.send_data_to_lambda(session, sample_data, lambda_url, PRECISION)

## Step 5: Configure Grafana

### Create Workspace or Use Existing Workspace

In [ ]:
WORKSPACE_NAME = "sample_app_workspace"
grafana_data_source_name = "Amazon Timestream for LiveAnalytics Sample Data Source"

workspace_id = grafana_helper.create_grafana_workspace(session, WORKSPACE_NAME,
                                                       "GrafanaSampleWorkspaceRole", DATABASE_NAME, TABLE_NAME)
service_account_token = grafana_helper.create_grafana_workspace_token(session, workspace_id)
grafana_workspace_url = grafana_helper.get_grafana_workspace_url(session, workspace_id)
grafana_helper.add_timestream_plugin(service_account_token, grafana_workspace_url)
grafana_helper.add_timestream_data_source(service_account_token, grafana_workspace_url,
                                          grafana_data_source_name, session.region_name, DATABASE_NAME, TABLE_NAME)

### Generate and Upload Grafana Dashboard

In [ ]:
dashboard = data_generator.generate_dashboard(grafana_data_source_name=grafana_data_source_name, database_name=DATABASE_NAME, table_name=TABLE_NAME)

# Write the dashboard JSON to a local file in order to
# upload manually
#with open('sample_app_dashboard.json', 'w') as f:
#    json.dump(dashboard, f)

dashboard_payload = {
    "dashboard": dashboard,
    "overwrite": True,  # Ensures replacement if it exists
    "id": None,
    "uid": None
}

create_dashboard_response = grafana_helper.create_dashboard(service_account_token, grafana_workspace_url, dashboard_payload)
create_dashboard_response.raise_for_status()
print(f"Dashboard deployed successfully")
print(f"Workspace login url: https://{grafana_workspace_url}/login")


### Add IAM Identity User to Grafana Workspace

This step must be done using the AWS management console. An IAM identity user must be added to the Grafana workspace. Only users added to the workspace will be able to log in to the workspace.

#### Create IAM Identity User

If you already have an IAM identity user you want to use to login to the workspace, skip to the next section.

1. [Go to the IAM Identity Center console](https://console.aws.amazon.com/singlesignon/home).
2. In the navigation pane, choose **Users**.
3. Choose **Add user**.
4. Input user details.
5. Choose **Next**.
6. Add the user to a group if you wish.
7. Choose **Next**.
8. Choose **Add user**.

#### Adding IAM Identity User to Workspace

1. [Go to the Amazon Managed Grafana console](https://console.aws.amazon.com/grafana/home).
2. In the navigation pane, choose **All workspaces**.
3. From the list of workspaces, choose the created workspace. By default, it is named `sample_app_workspace`.
4. in the **Authentication** tab, under **AWS IAM Identity Center (successor to AWS SSO)** choose **Assign new user or group**.
5. From the list of users, choose the user(s) you want to allow to login to the workspace and then choose **Assign users and groups**.
6. By default, users are added as a Viewer. If you want to allow your user to manage data sources in Grafana, select your user, then, in the **Action** dropdown menu, select **Make admin**.
7. Go to the login page as output by the previous cell and input your user's username and password to sign into the workspace.

### Viewing the Dashboard

1. Log in to the Amazon Managed Grafana workspace.
2. In the navigation pane, select **Dashboards**.
3. From the list of dashboards, select the deployed dashboard. By default, it is named `Amazon Timestream for LiveAnalytics Sample Dashboard`.
4. Adjust the time range as needed. By default, all data for the last 15 minutes is displayed. Timestamps are in UTC.
5. Measures are listed below the graph, select measures to display them.